# 00 — Preflight: is the Drive folder set up correctly?

**Run this first, once, before anybody starts a real notebook.** It takes about a minute, needs
no GPU, and tells you exactly what is present, what is missing, and what is nested at the wrong
depth — so nobody discovers a bad path 40 minutes into a training run.

It only reads; it changes nothing.

In [ ]:
from google.colab import drive
import sys, json
from pathlib import Path
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"    # <-- change if yours differs
root = Path(DRIVE_ROOT)
print("root exists:", root.exists(), "->", root)
if root.exists():
    for p in sorted(root.iterdir()):
        print("   ", p.name + ("/" if p.is_dir() else ""))

In [ ]:
# --- code/ : the repo snapshot the notebooks import from ---------------------
OK, BAD = [], []

def check(label, cond, detail=""):
    (OK if cond else BAD).append(label)
    print(f"  [{'OK ' if cond else 'FAIL'}] {label}{('  ' + detail) if detail else ''}")

print("code/")
code = root / "code"
check("code/colab_bootstrap.py", (code/"colab_bootstrap.py").exists())
check("code/src/compute_profile.py", (code/"src"/"compute_profile.py").exists())
check("code/src/iou.py", (code/"src"/"iou.py").exists())
n_src = len(list((code/"src").glob("*.py"))) if (code/"src").exists() else 0
check("code/src/ has modules", n_src >= 10, f"{n_src} .py files")

sys.path.insert(0, str(code))
import colab_bootstrap as CB
paths = CB.setup_paths(root)
print("  bootstrap imported OK")

In [ ]:
# --- inputs/ : manifest, invoice images, annotations -------------------------
print("inputs/")
import pandas as pd
man_p = paths.inputs / "invoice_manifest.csv"
check("inputs/invoice_manifest.csv", man_p.exists())
if man_p.exists():
    man = pd.read_csv(man_p)
    check("  manifest has 750 rows", len(man) == 750, f"{len(man)} rows")
    if "has_ground_truth" in man:
        cov = man.has_ground_truth.mean()
        check("  ground-truth coverage is 100%", cov > 0.99, f"{cov:.1%}")
    imgs = list((paths.inputs/"images").glob("*.jpg"))
    check("  inputs/images/ has 750 jpgs", len(imgs) == 750, f"{len(imgs)} files")
    missing = [r.image_path for r in man.head(50).itertuples()
               if not (root / r.image_path).exists()]
    check("  manifest paths resolve", not missing,
          f"{len(missing)} of first 50 missing" if missing else "sampled 50")

ann = list((paths.inputs/"annotations").glob("batch1_*.csv"))
check("inputs/annotations/batch1_*.csv", len(ann) == 3, f"{len(ann)} of 3")

In [ ]:
# --- inputs/datasets/ : the three real datasets ------------------------------
print("inputs/datasets/")
DATA = paths.inputs / "datasets"

def try_root(label, folder, markers):
    try:
        p = CB.resolve_dataset_root(DATA / folder, markers)
        extra = "" if p == DATA/folder else f"  (nested deeper: .../{p.name} - handled)"
        check(label, True, str(p.relative_to(DATA)) + extra)
        return p
    except FileNotFoundError as e:
        check(label, False, str(e).splitlines()[0])
        return None

OCR = try_root("datasets/ocr_multitype", "ocr_multitype",
               ["train/annotations", "val/annotations", "test/annotations"])
SIG = try_root("datasets/signatures", "signatures", ["images", "image_ids.csv"])
STA = try_root("datasets/stamps", "stamps", ["scans", "ground-truth-maps"])

if OCR:
    for sp, exp in [("train", 778), ("val", 97), ("test", 98)]:
        ni = len(list((OCR/sp/"images").glob("*")))
        na = len(list((OCR/sp/"annotations").glob("*.json")))
        check(f"  ocr {sp}", ni == exp and na == exp, f"{ni} images / {na} json (expect {exp})")

if SIG:
    try:
        d = CB.resolve_files_dir(SIG/"images", "*.png")
        npng = len(list(d.glob("*.png")))
        njpg = len(list(d.glob("*.jpeg"))) + len(list(d.glob("*.jpg")))
        # SignverOD ships a mix: 2,694 .png + 71 .jpeg = 2,765
        check("  signatures/images", npng + njpg >= 2700,
              f"{npng} png + {njpg} jpeg = {npng+njpg} (expect 2,765)")
    except FileNotFoundError as e:
        check("  signatures/images", False, str(e).splitlines()[0])
    for f in ["train.csv", "test.csv", "image_ids.csv", "categories.csv", "labelmap.txt"]:
        check(f"  signatures/{f}", (SIG/f).exists())

if STA:
    for sub, pat, exp in [("scans", "*.png", 427), ("ground-truth-maps", "*.png", 400),
                          ("info", "*.txt", 400)]:
        try:
            d = CB.resolve_files_dir(STA/sub, pat)
            n = len(list(d.glob(pat)))
            note = "" if d == STA/sub else f" (double-nested {sub}/{sub} - handled)"
            check(f"  stamps/{sub}", n >= exp*0.9, f"{n} files (expect {exp}){note}")
        except FileNotFoundError as e:
            check(f"  stamps/{sub}", False, str(e).splitlines()[0])

In [ ]:
# --- Verdict ------------------------------------------------------------------
print("\n" + "="*66)
print(f"PASSED {len(OK)}   FAILED {len(BAD)}")
print("="*66)
if BAD:
    print("\nFix these before running a member notebook:\n")
    for b in BAD:
        print("  -", b)
    print("\nSee inputs/datasets/COPY_MAP.md for the exact source -> destination mapping.")
    print("A 'nested deeper' NOTE above is NOT a failure - it is handled automatically.")
else:
    print("\nEverything checks out. Members can run their notebooks:")
    print("  02 Diana   needs datasets/signatures + datasets/stamps")
    print("  03 Jordan  needs datasets/ocr_multitype")
    print("  04 Damir   needs datasets/ocr_multitype + inputs/annotations")
    print("  05 Hessam  needs the others' published outputs (run last)")
    print("  01 Rolando is OPTIONAL - the manifest was already built locally")

## If something failed

| Symptom | Fix |
|---|---|
| `code/colab_bootstrap.py` missing | copy `colab/colab_bootstrap.py` into Drive `code/` |
| a dataset root not found | check `inputs/datasets/COPY_MAP.md`; the folder may be empty |
| counts lower than expected | the upload is probably still running — wait, then re-run |
| `stamps/scans` shows 0 | you copied the **outer** `scans` folder; the inner one holds the PNGs |

A **"nested deeper — handled"** note is not an error. It means you dragged the parent folder in
and the notebooks quietly compensate; no re-upload is needed.